In [6]:
import os
import json
from pprint import pprint
from typing import Dict,Any

CONFIG_FILE = "products_config.json"
if os.path.exists(CONFIG_FILE):
    with open(CONFIG_FILE, "r", encoding="utf-8") as file:
        config_data:Dict = json.load(file)
        #pprint(config_data.get("project_name"))
        print(f"✅ 成功載入設定檔！專案名稱：{config_data.get('project_name')}")
        print(f"📦 監控品類數量：{len(config_data.get('monitor_products', []))} 大類")
else:
    print(f"❌ 找不到設定檔 {CONFIG_FILE}")
        

✅ 成功載入設定檔！專案名稱：毛寶企業產品與競品價格每日監控系統
📦 監控品類數量：5 大類


###步驟2:單一賣場抓取---PCHOME 24h(API方式)

In [8]:
from playwright.async_api import async_playwright, Browser,BrowserContext,APIResponse
from typing import Dict,Any
from pprint import pprint
import urllib.parse

async def fetch_pchome(context:BrowserContext, keyword: str) -> Dict[str, Any]:
    """從 PChome 24h 購物抓取第一筆商品資訊 (API 方式)"""
    result = {
        "platform": "PChome 24h",
        "title": "未找到相關商品",
        "price": 0,
        "url": "",
        "status": "無結果"
    }
    encoded_kw= urllib.parse.quote(keyword)
    api_url = f"https://ecshweb.pchome.com.tw/search/v3.3/all/results?q={encoded_kw}&page=1"
    try:
        response:APIResponse = await context.request.get(api_url)
        if response.status == 200:
            data:Dict = await response.json()
            prods:list = data.get("prods",[])
            if prods:
                item:dict = prods[0]
                result['title'] = item.get("name", "未知的商品標題")
                result["price"] = int(item.get("price", 0))
                result["url"] = f"https://24h.pchome.com.tw/prod/{item.get('Id', '')}"
                result["status"] = "成功"
                return result
    except Exception as e:
        print(f"PChome 抓取失敗: {e}")
    return result
    
    
async with async_playwright() as p:
    browser:Browser = await p.firefox.launch(headless=True)
    context:BrowserContext = await browser.new_context()
    res:Dict[str, Any] = await fetch_pchome(context, "毛寶 洗衣槽")
    pprint(res)
    await browser.close()

NotImplementedError: 

In [9]:
import re
import urllib.parse
from typing import Dict, Any
from playwright.async_api import async_playwright, BrowserContext, Page,Locator

async def fetch_momo(context: BrowserContext, keyword: str) -> Dict[str, Any]:
    """從 momo購物網 抓取第一筆商品資訊 (DOM 解析)"""
    result = {
        "platform": "momo購物網",
        "title": "未找到相關商品",
        "price": 0,
        "url": "",
        "status": "無結果"
    }

    encoded_kw = urllib.parse.quote(keyword)
    url = f"https://www.momoshop.com.tw/search/searchShop.jsp?keyword={encoded_kw}"
    page: Page = await context.new_page()

    try:
        await page.goto(url, wait_until="domcontentloaded", timeout=15000)
        await page.wait_for_timeout(1000)
        cards:Locater = page.locator("div.listArea ul li, .prdListArea ul li")
        
        if await cards.count() > 0:
            card:Locator = cards.first
            title_loc:Locator = card.locator(".prdName, h3, .goodsName")
            price_loc:Locator = card.locator(".price, .money, .prdPrice")
            link_loc:Locator = card.locator("a.goods-img-url, a.prdName, a").first

            title:str = await title_loc.first.inner_text() if await title_loc.count() > 0 else ""
            price_text:str = await price_loc.first.inner_text() if await price_loc.count() > 0 else ""
            href:str = await link_loc.get_attribute("href") if await link_loc.count() > 0 else "" # type: ignore

            digits = re.sub(r"[^\d]", "", price_text)
            price = int(digits) if digits else 0

            if href and not href.startswith("http"):
                href = f"https://www.momoshop.com.tw{href}"

            if title:
                result["title"] = title.strip()
                result["price"] = price
                result["url"] = href
                result["status"] = "成功"
    except Exception as e:
        print(f"momo 抓取失敗: {e}")
    finally:
        await page.close()

    return result

# 單獨測試 fetch_momo
async with async_playwright() as p:
    browser:Browser = await p.firefox.launch(headless=True)
    context:BrowserContext = await browser.new_context(user_agent="Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36")
    res:Dict = await fetch_momo(context, "毛寶 貼身衣物手洗精")
    pprint(res)
    await browser.close()

NotImplementedError: 

In [10]:
import re
import urllib.parse
from typing import Dict, Any
from playwright.async_api import async_playwright, BrowserContext, Page

async def fetch_yahoo(context: BrowserContext, keyword: str) -> Dict[str, Any]:
    """從 Yahoo購物中心 抓取第一筆商品資訊 (DOM 解析)"""
    result = {
        "platform": "Yahoo購物中心",
        "title": "未找到相關商品",
        "price": 0,
        "url": "",
        "status": "無結果"
    }

    encoded_kw = urllib.parse.quote(keyword)
    url = f"https://tw.buy.yahoo.com/search/product?p={encoded_kw}"
    page: Page = await context.new_page()

    try:
        await page.goto(url, wait_until="domcontentloaded", timeout=15000)
        await page.wait_for_timeout(1200)
        cards = page.locator("a[href*='/gdsale/']")

        if await cards.count() > 0:
            card = cards.first
            href = await card.get_attribute("href")
            txt = await card.inner_text()
            lines = [l.strip() for l in txt.split("\n") if l.strip()]

            title = ""
            price = 0
            for l in lines:
                if l.startswith("$"):
                    digits = re.sub(r"[^\d]", "", l)
                    if digits and price == 0:
                        price = int(digits)
                elif l not in ["比較", "找相似", "活動", "券", "限時下殺", "折扣"] and not title:
                    title = l

            if title:
                result["title"] = title
                result["price"] = price
                result["url"] = href or ""
                result["status"] = "成功"
    except Exception as e:
        print(f"Yahoo 抓取失敗: {e}")
    finally:
        await page.close()

    return result

# 單獨測試 fetch_yahoo
async with async_playwright() as p:
    browser = await p.chromium.launch(headless=True)
    context = await browser.new_context(user_agent="Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36")
    res = await fetch_yahoo(context, "毛寶 貼身衣物手洗精")
    print(res)
    await browser.close()

NotImplementedError: 